# 01 — Cohort Retention

D1/D7/D30 retention by signup ISO week. Portfolio chart: retention heatmap.

**Run:**
```bash
pip install -r requirements-dev.txt
jupyter notebook analysis/01_cohort_retention.ipynb
```

**Data:** latest export ZIP in `analysis/exports/` or live `studybuddy.db`

In [ ]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "analysis" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import asyncio
from analysis.lib.pa_loaders import find_latest_export, load_export_zip, load_sqlite
from analysis.lib.pa_charts import plot_retention_heatmap
from db import get_db, init_db
from services import AnalyticsService

OUTPUT_DIR = os.path.join(REPO_ROOT, "analysis", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
async def load_retention():
    export = find_latest_export()
    if export:
        users, events, meta = load_export_zip(export)
        print(f"Loaded export: {export.name} (users={len(users)})")
    db_path = os.path.join(REPO_ROOT, "studybuddy.db")
    if os.path.exists(db_path):
        db = await get_db(db_path)
        await init_db(db)
        analytics = AnalyticsService(db)
        data = await analytics.compute_cohort_retention()
        await db.close()
        return data
    return {"cohorts": [], "total_users": 0}

retention = asyncio.run(load_retention())
retention

In [ ]:
cohorts = retention.get("cohorts", [])
if cohorts:
    fig = plot_retention_heatmap(cohorts, title="Palph Cohort Retention")
    out = os.path.join(OUTPUT_DIR, "retention_heatmap.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
else:
    print("No cohort data yet — run after Week 1 launch.")

## Key findings (fill after run)

1. Best cohort: ___ (D7 = ___%)
2. Worst cohort: ___ (D7 = ___%)
3. Recommendation: ___